In [11]:
import pandas as pd
import requests
import json
import time

try:
    input_df = pd.read_csv("uniprotkb_results.csv")
    print(f"Successfully loaded {len(input_df)} records from CSV.")
except FileNotFoundError:
    print("CSV not found. Generating 6 sample values for development/testing...")
    mock_data = {
        'Entry': ['P04637', 'P12345', 'P69905', 'P00533', 'P35555', 'Q9BYF1'],
        'Sequence': [
            'MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP',
            'MKAILVVLLYTFATANADTLCIGYHANNSTDTVDTVLEKNVTVTHSVNLLEDSHNGKLCL',
            'MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG',
            'MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFEDHFLSLQRMFNNCEV',
            'MLDDRARMEAAKKKEKVEQILEEFQLQEEDLKKVMSRAQKGKRLPGLTGVDVGMGRDLSK',
            'MAAARRSGRGRRGRGRGRGRGLRGRGRGRGRGRGRGRGRGRGRGRGRGRGRGRGRGRGRG'
        ]
    }
    input_df = pd.DataFrame(mock_data)
    print("Mock dataset ready for analysis.")

input_df.head()

CSV not found. Generating 6 sample values for development/testing...
Mock dataset ready for analysis.


,Entry,Sequence
0,P04637,MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLS...
1,P12345,MKAILVVLLYTFATANADTLCIGYHANNSTDTVDTVLEKNVTVTHS...
2,P69905,MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPH...
3,P00533,MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFED...
4,P35555,MLDDRARMEAAKKKEKVEQILEEFQLQEEDLKKVMSRAQKGKRLPG...


In [12]:
def fetch_interpro_annotations(uniprot_id):
    """Fetches functional domains and PDB templates from InterPro."""
    api_url = f"https://www.ebi.ac.uk/interpro/api/entry/interpro/protein/uniprot/{uniprot_id}"
    annotations = []
    try:
        response = requests.get(api_url, timeout=20)
        if response.status_code == 200:
            data = response.json().get('results', [])
            for entry in data:
                metadata = entry.get('metadata', {})
                if metadata.get('type') in ['domain', 'family']:
                    pdbs = [xref.get('id') for xref in entry.get('xrefs', []) if xref.get('database') == 'PDB']

                    locs = entry.get('proteins', [{}])[0].get('entry_protein_locations', [])
                    for loc in locs:
                        for frag in loc.get('fragments', []):
                            annotations.append({
                                'domain_id': metadata.get('accession'),
                                'domain_name': metadata.get('name'),
                                'start': frag.get('start'),
                                'end': frag.get('end'),
                                'pdb_template': pdbs[0] if pdbs else None,
                                'source': 'InterPro'
                            })
        return annotations
    except Exception as e:
        print(f"InterPro API error for {uniprot_id}: {e}")
        return []

def fetch_cath_annotations(uniprot_id):
    """Fetches structural classifications from CATH."""
    api_url = f"https://www.cathdb.info/api/rest/v4/uniprot/{uniprot_id}"
    annotations = []
    try:
        response = requests.get(api_url, timeout=15)
        if response.status_code == 200:
            matches = response.json().get('data', {}).get('matches', [])
            for m in matches:
                annotations.append({
                    'domain_id': m.get('cath_id'),
                    'domain_name': m.get('class_name', 'Structural Domain'),
                    'start': m.get('start'),
                    'end': m.get('end'),
                    'pdb_template': None,
                    'source': 'CATH'
                })
        return annotations
    except Exception as e:
        print(f"CATH API error for {uniprot_id}: {e}")
        return []

In [13]:
enriched_data = []

for index, row in input_df.iterrows():
    uniprot_id = row['Entry']
    full_sequence = row['Sequence']

    print(f"Processing {uniprot_id}...")

    domain_hits = fetch_interpro_annotations(uniprot_id) + fetch_cath_annotations(uniprot_id)

    if not domain_hits:
        enriched_data.append({
            'uniprot_id': uniprot_id,
            'domain_id': 'FULL',
            'domain_name': 'Full Protein Sequence',
            'coords': f"1-{len(full_sequence)}",
            'sliced_sequence': full_sequence,
            'pdb_template': None,
            'source': 'None'
        })
    else:
        for domain in domain_hits:
            start, end = domain['start'], domain['end']
            sliced_seq = full_sequence[start-1 : end]

            enriched_data.append({
                'uniprot_id': uniprot_id,
                'domain_id': domain['domain_id'],
                'domain_name': domain['domain_name'],
                'coords': f"{start}-{end}",
                'sliced_sequence': sliced_seq,
                'pdb_template': domain['pdb_template'],
                'source': domain['source']
            })

    time.sleep(0.4)

output_df = pd.DataFrame(enriched_data)
output_df.head()

Processing P04637...
Processing P12345...
Processing P69905...
Processing P00533...
Processing P35555...
Processing Q9BYF1...


,uniprot_id,domain_id,domain_name,coords,sliced_sequence,pdb_template,source
0,P04637,IPR002117,p53 tumour suppressor family,3-369,EPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPD...,None,InterPro
1,P04637,IPR010991,"p53, tetramerisation domain",319-357,,None,InterPro
2,P04637,IPR011615,"p53, DNA-binding domain",100-288,,None,InterPro
3,P04637,IPR013872,"p53, transactivation domain",6-30,SDPSVEPPLSQETFSDLWKLLPENN,None,InterPro
4,P04637,IPR040926,"Cellular tumor antigen p53, transactivation do...",35-59,LPSQAMDDLMLSPDDIEQWFTEDPG,None,InterPro


In [14]:
output_path = "domain_analysis_output.json"
output_df.to_json(output_path, orient="records", indent=4)

print(f"Analysis complete. Data exported to {output_path} for structural prediction.")

Analysis complete. Data exported to domain_analysis_output.json for structural prediction.
